# 05 — Habitudes temporelles de notation : le chunk K=5 est-il le bon grain ?

**Question.** Le chunk de taille `K=5` (`data_jepa.to_chunks`) a toujours été un choix *arbitraire*.
Les timestamps MovieLens sont des dates de **notation en rafales**, pas de visionnage. Ce notebook
mesure la structure réelle de ces rafales pour trancher trois points de design :

1. Existe-t-il une **taille de rafale naturelle** — et vaut-elle ~5 ?
2. Y a-t-il une **vallée nette** dans les écarts (un seuil de session non-arbitraire) ?
3. Le K=5 fixe **respecte-t-il les frontières de session**, ou coupe-t-il au milieu d'une rafale
   (ce qui violerait l'hypothèse « chunk = ensemble d'un même moment, sans ordre interne ») ?

Tout part de `sequences['timestamps']` (unix s, déjà trié, tronqué aux 500 derniers). Aucune
relecture du `rating.csv` de 690 Mo. Logique dans `src/timing_stats.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 20)

from src.data_prep import load_all
from src import timing_stats as T

seq, genome, maps = load_all()
print(f'{len(seq):,} users, longueur médiane {int(seq.length.median())}')
SAMPLE = 40000   # users échantillonnés pour les stats de session (None = tous)

## 1. Écarts entre notations consécutives

L'objet fondamental : pour chaque user, le temps (secondes) entre deux notes qui se suivent.
Comme l'écrasante majorité des écarts sont *minuscules*, un histogramme brut est illisible ;
on regarde la **répartition par tranches de temps** (part des écarts en dessous de chaque seuil).

In [ ]:
g = T.all_gaps(seq, sample=None)   # tous les users, c'est bon marché
print(f'{g.size:,} écarts')
for q in (0.5, 0.75, 0.9, 0.99, 0.999):
    v = np.quantile(g, q); print(f'  p{q*100:>5.1f} : {T.human_seconds(v):>8}  ({v:,.0f} s)')

edges = [0, 1, 60, 600, 3600, 86400, 604800, np.inf]
labels = ['= 0 s\n(même seconde)', '< 1 min', '< 10 min', '< 1 h', '< 1 j', '< 1 sem', '>= 1 sem']
frac = [np.mean((g >= edges[i]) & (g < edges[i+1])) for i in range(len(labels))]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, frac, color=sns.color_palette('flare', len(labels)))
ax.set_ylabel('part des écarts'); ax.set_title('Répartition des écarts entre notations consécutives')
for b, f in zip(bars, frac):
    ax.text(b.get_x()+b.get_width()/2, f+0.01, f'{f:.1%}', ha='center', fontsize=9)
ax.set_ylim(0, max(frac)*1.15); plt.tight_layout(); plt.show()

**Lecture.** Si une énorme part des écarts est *exactement 0 s*, l'ordre intra-rafale est en grande
partie **arbitraire** (notes posées d'un bloc) → cela justifie l'encodeur de chunk **sans encodage
positionnel**. La présence (ou non) d'un *trou* entre deux tranches indique s'il existe un seuil de
session naturel.

## 2. Segmentation en sessions (rafales)

Une **session** = suite maximale de notes dont les écarts internes sont `< tau`. On fait varier `tau`
et on regarde la distribution des **tailles de session**. C'est la réponse directe à « y a-t-il une
taille naturelle ~5 ? ». La colonne `%_notes_en_sessions>=K` dit si K=5 est trop grand (< 50 %) ou
trop petit (les rafales le dépassent largement) pour la masse des notes.

In [ ]:
summary = T.burst_size_summary(seq, sample=SAMPLE)
summary.style.format({'moyenne':'{:.1f}','médiane':'{:.0f}','p90':'{:.0f}',
    '%_taille_1':'{:.1%}','%_==K(5)':'{:.1%}','%_<K':'{:.1%}','%_>K':'{:.1%}',
    '%_notes_en_sessions>=K':'{:.1%}'})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, tau) in zip(axes, [('10 min', 600), ('1 h', 3600)]):
    s = T.burst_sizes(seq, tau, sample=SAMPLE)
    s = np.clip(s, 1, 40)
    ax.hist(s, bins=np.arange(1, 42)-0.5, color='#4C72B0')
    ax.axvline(5, color='crimson', ls='--', lw=2, label='K = 5')
    ax.axvline(np.median(T.burst_sizes(seq, tau, sample=SAMPLE)), color='green', ls=':', lw=2, label='médiane')
    ax.set_title(f'Tailles de session (seuil {label})'); ax.set_xlabel('films par session (clip 40)')
    ax.legend()
plt.tight_layout(); plt.show()

## 3. Notes à la même seconde : l'ordre intra-rafale existe-t-il ?

On isole les paquets de notes partageant **exactement le même timestamp** (seuil `tau=1 s`).
Un paquet médian > 1 confirme que l'ordre *à l'intérieur* d'un chunk est un artefact de tri, pas un
signal — argument central pour l'encodeur de chunk sans position.

In [ ]:
stats = T.same_timestamp_stats(seq, sample=SAMPLE)
for k, v in stats.items():
    print(f'  {k:<32} {v:.3f}' if isinstance(v, float) else f'  {k:<32} {v}')

## 4. Ce que le K fixe fait aux rafales réelles

Deux façons pour K=5 de mal tomber :
- **chunk impur** : ses 5 films chevauchent une frontière de session (un gros trou interne) → il
  mélange deux moments, ce que l'archi suppose ne PAS arriver ;
- **fragmentation** : la session est bien plus grande que 5, donc une même rafale est hachée en
  plusieurs chunks que le Transformer temporel traite comme des pas successifs (alors que l'ordre
  y est arbitraire).

`chunk_purity_summary` mesure l'impureté en fonction de K (seuil de session `tau`).

In [ ]:
purity = T.chunk_purity_summary(seq, tau=3600, sample=SAMPLE)
display(purity.style.format({'%_chunks_propres(<tau)':'{:.1%}','%_chunks_impurs(>=tau)':'{:.1%}',
    'médiane_maxgap_h':'{:.2f}','p90_maxgap_j':'{:.3f}'}))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(purity['K'], purity['%_chunks_impurs(>=tau)']*100, 'o-', color='crimson')
ax.axvline(5, color='gray', ls='--', label='K = 5 actuel')
ax.set_xlabel('K (taille de chunk)'); ax.set_ylabel('% chunks impurs (chevauchent une session, seuil 1 h)')
ax.set_title('Coût d\'un K plus grand : impureté des chunks'); ax.legend()
plt.tight_layout(); plt.show()

## 5. Lecture & implications de design

*(À compléter/confirmer après exécution — les chiffres ci-dessous sont ceux d'un run sample=40000.)*

- **Pas de taille naturelle ~5.** Les rafales sont soit des paquets *même-seconde* de médiane ~3,
  soit des sessions bien plus grandes (médiane ~33 à un seuil de 1 h). K=5 tombe dans un entre-deux.
- **~91 % des écarts consécutifs sont EXACTEMENT 0 s** → l'ordre intra-chunk est très largement
  arbitraire. **Ceci VALIDE fortement** l'encodeur de chunk sans encodage positionnel.
- **K=5 ne coupe presque jamais une frontière de session** (~98 % de chunks « propres » à 1 h) —
  non parce que 5 est bien choisi, mais parce que **les sessions sont bien plus grandes que 5**.
  Conséquence : K=5 **fragmente** une rafale (ex. une session de 33 → ~7 chunks), et le Transformer
  temporel voit surtout des transitions *intra-rafale* (ordre arbitraire), pas des transitions entre
  vrais moments. C'est cohérent avec le signal temporel court/faible du notebook 04.

**Pistes de design à considérer :**
1. **Chunking par session** (taille variable, coupe sur gap ≥ tau) plutôt que K fixe → chaque pas
   temporel = un vrai moment. Nécessite un cap (sessions jusqu'à 500) et un padding de chunk.
2. **Le gap comme feature** (déjà en réserve dans `sequences['timestamps']`) : donner au modèle la
   frontière de session plutôt que la déduire.
3. Si on garde K fixe, l'assumer explicitement comme un **sous-échantillonnage** de sessions, pas
   comme une unité temporelle.